# 03: Benchmark Comparison

This notebook compares the performance of all portfolio optimization algorithms implemented in the prototype:
- **Exact Enumeration**: Brute-force ground truth for instances up to ~12 assets
- **MVO (Continuous)**: Markowitz mean-variance optimization (convex)
- **Genetic Algorithm (GA)**: Population-based discrete optimizer
- **Simulated Annealing (SA)**: Temperature-based discrete optimizer
- **QAOA (Quantum)**: Parameterized quantum circuit on ideal simulator

All algorithms solve the same objective for fair comparison.

In [1]:
print("Hello, World!")

Hello, World!


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import portfolio optimization modules
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from portfolio_opt.data import (
    build_default_price_data,
    clean_price_frame,
    estimate_expected_returns,
    estimate_covariance,
)
from portfolio_opt.portfolio import (
    discrete_objective,
    build_discrete_qubo,
    solve_mvo,
    exact_enumeration,
)
from portfolio_opt.ga import genetic_algorithm
from portfolio_opt.sa import simulated_annealing
from portfolio_opt.qa import run_qaoa_on_qubo
from portfolio_opt.metrics import sharpe_ratio, sortino_ratio, max_drawdown

print("All modules loaded successfully.")

All modules loaded successfully.


## Data Preparation

In [3]:
# Build deterministic default price data (4 assets, 120 days)
prices = build_default_price_data(n_assets=4, n_days=120, seed=42)
print(f"Price data shape: {prices.shape}")
print(f"Assets: {list(prices.columns)}")
print(f"\nFirst 5 rows:")
print(prices.head())

# Clean missing values
prices_clean, clean_report = clean_price_frame(prices)
print(f"\nData cleaning report: {clean_report}")

# Calculate returns, expected returns, and covariance
returns = prices_clean.pct_change().dropna()
mu = estimate_expected_returns(returns)
cov = estimate_covariance(returns)

print(f"\nExpected returns (annualized):")
print(mu)
print(f"\nCovariance matrix:")
print(cov)

Price data shape: (120, 4)
Assets: ['AAPL', 'MSFT', 'AMZN', 'GOOGL']

First 5 rows:
                  AAPL        MSFT       AMZN       GOOGL
2020-01-02  100.000000  150.000000  90.000000  110.000000
2020-01-03   99.545626  149.394314  89.545539  109.215687
2020-01-06  101.662798  152.133097  91.712911  109.604966
2020-01-07  101.554345  151.977228  91.611199  106.931147
2020-01-08  102.971307  153.799797  93.069019  104.262041

Data cleaning report: {'missing_cells': 0, 'remaining_missing_cells': 0, 'cleaned_rows': 120, 'columns': ['AAPL', 'MSFT', 'AMZN', 'GOOGL'], 'missing_by_column': {'AAPL': 0, 'MSFT': 0, 'AMZN': 0, 'GOOGL': 0}, 'status': 'missing values were filled deterministically and reported'}

Expected returns (annualized):
AAPL     0.574522
MSFT     0.472719
AMZN     0.676325
GOOGL    0.750668
dtype: float64

Covariance matrix:
           AAPL      MSFT      AMZN     GOOGL
AAPL   0.051789  0.044884  0.058694  0.001826
MSFT   0.044884  0.038899  0.050868  0.001582
AMZN   0.05

## Algorithm Configuration

In [4]:
# Problem parameters
k = 2  # Select 2 assets from 4
lambda_risk = 1.0  # Risk aversion parameter
random_seed = 42

print(f"Problem configuration:")
print(f"  Assets to select (k): {k} from {len(mu)}")
print(f"  Risk aversion (λ): {lambda_risk}")
print(f"  Random seed: {random_seed}")

# Algorithm-specific parameters
ga_params = {
    'population_size': 30,
    'generations': 50,
    'mutation_rate': 0.1,
    'elite_fraction': 0.2,
    'seed': random_seed,
}

sa_params = {
    'iterations': 500,
    'initial_temp': 1.0,
    'cooling_factor': 0.995,
    'seed': random_seed,
}

qaoa_params = {
    'depth': 1,
    'shots': 512,
    'seed': random_seed,
}

print(f"\nGA parameters: {ga_params}")
print(f"SA parameters: {sa_params}")
print(f"QAOA parameters: {qaoa_params}")

Problem configuration:
  Assets to select (k): 2 from 4
  Risk aversion (λ): 1.0
  Random seed: 42

GA parameters: {'population_size': 30, 'generations': 50, 'mutation_rate': 0.1, 'elite_fraction': 0.2, 'seed': 42}
SA parameters: {'iterations': 500, 'initial_temp': 1.0, 'cooling_factor': 0.995, 'seed': 42}
QAOA parameters: {'depth': 1, 'shots': 512, 'seed': 42}


## Run All Algorithms

In [5]:
results = {}

# 1. Exact Enumeration (ground truth)
print("Running Exact Enumeration...")
exact_result = exact_enumeration(mu, cov, k, risk_aversion=lambda_risk)
x_exact = exact_result["x"]
obj_exact = exact_result["objective"]
results['Exact'] = {
    'x': x_exact,
    'objective': obj_exact,
    'feasible': True,
    'selected': list(np.where(x_exact)[0]),
}
print(f"  Objective: {obj_exact:.6f}")
print(f"  Selected indices: {results['Exact']['selected']}")

# 2. MVO (Continuous baseline)
print("\nRunning MVO (Continuous)...")
w_mvo = solve_mvo(mu, cov, risk_aversion=lambda_risk, k=None)
x_mvo = np.zeros(len(mu))
x_mvo[np.argsort(w_mvo)[-k:]] = 1  # Select top k weights
obj_mvo = discrete_objective(x_mvo, mu, cov, k, risk_aversion=lambda_risk)
results['MVO'] = {
    'x': x_mvo,
    'objective': obj_mvo,
    'feasible': True,
    'selected': list(np.where(x_mvo)[0]),
}
print(f"  Objective: {obj_mvo:.6f}")
print(f"  Selected indices: {results['MVO']['selected']}")

# 3. Genetic Algorithm
print("\nRunning Genetic Algorithm...")
ga_result = genetic_algorithm(mu, cov, k, **ga_params, risk_aversion=lambda_risk)
results['GA'] = {
    'x': ga_result['x'],
    'objective': ga_result['objective'],
    'feasible': ga_result['x'].sum() == k,
    'selected': list(np.where(ga_result['x'])[0]),
    'generations': ga_result.get('generations', 'N/A'),
}
print(f"  Objective: {ga_result['objective']:.6f}")
print(f"  Selected indices: {results['GA']['selected']}")

# 4. Simulated Annealing
print("\nRunning Simulated Annealing...")
sa_result = simulated_annealing(mu, cov, k, **sa_params, risk_aversion=lambda_risk)
results['SA'] = {
    'x': sa_result['x'],
    'objective': sa_result['objective'],
    'feasible': sa_result['x'].sum() == k,
    'selected': list(np.where(sa_result['x'])[0]),
    'iterations': sa_result.get('iterations', 'N/A'),
}
print(f"  Objective: {sa_result['objective']:.6f}")
print(f"  Selected indices: {results['SA']['selected']}")

# 5. QAOA
print("\nRunning QAOA...")
qaoa_result = run_qaoa_on_qubo(mu, cov, k, **qaoa_params, risk_aversion=lambda_risk)
results['QAOA'] = {
    'x': qaoa_result['x'],
    'objective': qaoa_result['objective'],
    'feasible': qaoa_result['feasible'],
    'selected': list(np.where(qaoa_result['x'])[0]),
    'feasibility_rate': qaoa_result['feasibility_rate'],
}
print(f"  Objective: {qaoa_result['objective']:.6f}")
print(f"  Selected indices: {results['QAOA']['selected']}")
print(f"  Feasibility rate: {qaoa_result['feasibility_rate']:.2%}")

Running Exact Enumeration...
  Objective: 0.684667
  Selected indices: [np.int64(2), np.int64(3)]

Running MVO (Continuous)...
  Objective: 0.684667
  Selected indices: [np.int64(2), np.int64(3)]

Running Genetic Algorithm...
  Objective: 0.684667
  Selected indices: [np.int64(2), np.int64(3)]

Running Simulated Annealing...
  Objective: 0.684667
  Selected indices: [np.int64(2), np.int64(3)]

Running QAOA...
  Objective: 0.684667
  Selected indices: [np.int64(2), np.int64(3)]
  Feasibility rate: 28.61%


## Summary Comparison

In [ ]:
# Create comparison table
comparison_data = []
for algo, result in results.items():
    gap_from_exact = (obj_exact - result['objective']) / obj_exact * 100 if obj_exact != 0 else 0
    comparison_data.append({
        'Algorithm': algo,
        'Objective': result['objective'],
        'Gap from Exact (%)': gap_from_exact,
        'Feasible': result['feasible'],
        'Selected Assets': str(result['selected']),
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("ALGORITHM COMPARISON SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

## Visualization: Objective Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Objective values
algo_names = list(results.keys())
objectives = [results[a]['objective'] for a in algo_names]
colors = ['green', 'blue', 'orange', 'purple', 'red']

ax = axes[0]
bars = ax.bar(algo_names, objectives, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Portfolio Objective', fontsize=12, fontweight='bold')
ax.set_title('Objective Value Comparison\n(Higher is Better)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
for bar, obj in zip(bars, objectives):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{obj:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Optimality gap
gaps = [(obj_exact - obj) / obj_exact * 100 if obj_exact != 0 else 0 for obj in objectives]
ax = axes[1]
bars = ax.bar(algo_names, gaps, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Optimality Gap (%)', fontsize=12, fontweight='bold')
ax.set_title('Optimality Gap from Exact Solution\n(Lower is Better)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
for bar, gap in zip(bars, gaps):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{gap:.2f}%', ha='center', va='bottom' if gap >= 0 else 'top', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print("\nVisualization created successfully.")

## Convergence Curves (for iterative algorithms)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GA convergence
if 'history' in ga_result:
    ax = axes[0]
    ax.plot(ga_result['history'], marker='o', markersize=4, linewidth=2, color='orange', label='Best Objective')
    ax.axhline(y=obj_exact, color='green', linestyle='--', linewidth=2, label='Exact (Ground Truth)')
    ax.set_xlabel('Generation', fontsize=11, fontweight='bold')
    ax.set_ylabel('Best Objective', fontsize=11, fontweight='bold')
    ax.set_title(f'Genetic Algorithm Convergence\n(Population={ga_params["population_size"]}, Generations={ga_params["generations"]})', 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=10)
else:
    ax = axes[0]
    ax.text(0.5, 0.5, 'No convergence history available', ha='center', va='center', fontsize=12)
    ax.axis('off')

# SA convergence
if 'history' in sa_result:
    ax = axes[1]
    ax.plot(sa_result['history'], marker='s', markersize=4, linewidth=2, color='purple', label='Best Objective')
    ax.axhline(y=obj_exact, color='green', linestyle='--', linewidth=2, label='Exact (Ground Truth)')
    ax.set_xlabel('Iteration', fontsize=11, fontweight='bold')
    ax.set_ylabel('Best Objective', fontsize=11, fontweight='bold')
    ax.set_title(f'Simulated Annealing Convergence\n(Iterations={sa_params["iterations"]}, Cooling={sa_params["cooling_factor"]})', 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=10)
else:
    ax = axes[1]
    ax.text(0.5, 0.5, 'No convergence history available', ha='center', va='center', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()
print("\nConvergence curves displayed.")

## Key Insights

1. **Exact Enumeration** provides the ground truth and is optimal for this small instance (k=2 from 4 assets).

2. **MVO** (continuous) provides a fast baseline by selecting assets with highest MVO weights. It's not guaranteed to be optimal for the discrete cardinality-constrained problem but serves as a comparison point.

3. **Genetic Algorithm** uses population-based search with elite preservation and converges to high-quality solutions. Population diversity helps escape local optima.

4. **Simulated Annealing** uses swap-based moves that preserve cardinality, showing smooth convergence with temperature cooling.

5. **QAOA** leverages quantum circuit parameterization and sampling. The feasibility rate reflects how many sampled bitstrings satisfied the cardinality constraint k=2.

### Practical Takeaway
For this 4-asset problem, all algorithms find near-optimal or optimal solutions. At larger scales (n > 12), exact enumeration becomes intractable, and quantum/metaheuristic methods become necessary.